# Deception Detection using Face and Audio — Google Colab Training

**Paper**: Audio-Visual Deception Detection (ICCV 2023)  
**Dataset**: DOLOS (from "Would I Lie to You?")  
**GPU Required**: Yes (Runtime → Change runtime type → T4 GPU)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Clone Repository

In [2]:
!git clone https://github.com/PreciousTechnologies/Deception-Detection-using-Face-and-audio.git
%cd Deception-Detection-using-Face-and-audio

Cloning into 'Deception-Detection-using-Face-and-audio'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 158 (delta 51), reused 157 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 3.82 MiB | 8.91 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/Deception-Detection-using-Face-and-audio


## Step 2: Install Dependencies

In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install opencv-python-headless scikit-learn pandas matplotlib seaborn tqdm pillow librosa

Looking in indexes: https://download.pytorch.org/whl/cu124


## Step 3: Verify GPU

In [5]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime → Change runtime type → T4 GPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Step 4: Download DOLOS Dataset

The dataset is hosted by ROSE Lab, NTU Singapore. This downloads the preprocessed audio (.wav) and face frames (.jpg) needed for training.

In [6]:
import os
import gdown

# Create data directories
os.makedirs('data/audio_files', exist_ok=True)
os.makedirs('data/face_frames', exist_ok=True)
os.makedirs('data/protocols', exist_ok=True)

# Copy training protocols from the repo
!cp DOLOSDATA/DOLOS/Training_Protocols/*.csv data/protocols/

print("Data directories created.")
print("Protocols copied:")
!ls data/protocols/

Data directories created.
Protocols copied:
female.csv  male.csv   test_fold1.csv  test_fold3.csv	train_fold2.csv
long.csv    short.csv  test_fold2.csv  train_fold1.csv	train_fold3.csv


In [ ]:
# ============================================================
# OPTION A: Download from Google Drive (if available)
# Uncomment and replace FILE_ID with your actual Google Drive file IDs
# ============================================================
# gdown.download(id='YOUR_AUDIO_FILE_ID', output='data/audio_files.zip', quiet=False)
# gdown.download(id='YOUR_FRAMES_FILE_ID', output='data/face_frames.zip', quiet=False)
# !unzip -q data/audio_files.zip -d data/
# !unzip -q data/face_frames.zip -d data/

# ============================================================
# OPTION B: Download raw videos and preprocess locally
# Uncomment the cells below to download and preprocess
# ============================================================
# !pip install yt-dlp
# from scripts.dataloader.audio_visual_dataset import AudioVisualDataset
# Run the video downloader and preprocessor (see AV-Data-Processing/)

In [ ]:
# ============================================================
# OPTION C: Mount Google Drive (RECOMMENDED)
# Upload your audio_files/ and face_frames/ folders to Google Drive
# then mount and copy them over
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# After uploading your data to Google Drive, update these paths:
# !cp -r /content/drive/MyDrive/DOLOS_DATA/audio_files/* data/audio_files/
# !cp -r /content/drive/MyDrive/DOLOS_DATA/face_frames/* data/face_frames/

print("\nIMPORTANT: Uncomment the copy commands above and point them to your Drive data.")

## Step 5: Verify Dataset

In [ ]:
import pandas as pd

# Check protocols
train_df = pd.read_csv('data/protocols/train_fold1.csv')
test_df = pd.read_csv('data/protocols/test_fold1.csv')
print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Label distribution (train):\n{train_df.iloc[:, 1].value_counts()}")

# Check audio files
audio_files = [f for f in os.listdir('data/audio_files') if f.endswith('.wav')]
print(f"\nAudio files found: {len(audio_files)}")

# Check face frame directories
if os.path.exists('data/face_frames'):
    frame_dirs = [d for d in os.listdir('data/face_frames') if os.path.isdir(os.path.join('data/face_frames', d))]
    print(f"Face frame directories found: {len(frame_dirs)}")
else:
    print("WARNING: face_frames directory is empty or missing!")

## Step 6: Preprocess — Extract Face Frames from Videos

Run this if you downloaded raw videos instead of preprocessed data.

In [ ]:
# ============================================================
# Only run this if you have raw .mp4 videos in DOLOSDATA/raw_videos/
# Skip if you already have face_frames/ and audio_files/
# ============================================================
import subprocess

# Extract audio from videos
raw_video_dir = 'DOLOSDATA/raw_videos'
audio_out_dir = 'data/audio_files'
face_out_dir = 'data/face_frames'

if os.path.exists(raw_video_dir):
    videos = [f for f in os.listdir(raw_video_dir) if f.endswith('.mp4')]
    print(f"Found {len(videos)} raw videos")

    # Extract audio
    for v in videos:
        out_wav = os.path.join(audio_out_dir, v.replace('.mp4', '.wav'))
        if not os.path.exists(out_wav):
            subprocess.run(['ffmpeg', '-i', os.path.join(raw_video_dir, v),
                          '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
                          out_wav], capture_output=True)
    print("Audio extraction complete.")

    # Extract face frames using MTCNN
    !pip install facenet-pytorch
    from facenet_pytorch import MTCNN
    from PIL import Image
    import torchvision.transforms as T

    mtcnn = MTCNN(keep_all=True, device='cuda' if torch.cuda.is_available() else 'cpu')
    transform = T.Compose([T.Resize((160, 160))])

    for v in videos:
        clip_name = v.replace('.mp4', '')
        frame_dir = os.path.join(face_out_dir, clip_name)
        if os.path.exists(frame_dir) and len(os.listdir(frame_dir)) > 0:
            continue
        os.makedirs(frame_dir, exist_ok=True)

        # Extract frames with OpenCV
        import cv2
        cap = cv2.VideoCapture(os.path.join(raw_video_dir, v))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = 0
        saved_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % max(1, int(fps // 4)) == 0:  # ~4 fps
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                face = mtcnn(Image.fromarray(frame_rgb))
                if face is not None:
                    face_pil = T.ToPILImage()(face)
                    face_pil.save(os.path.join(frame_dir, f'{saved_count:04d}.jpg'))
                    saved_count += 1
            frame_count += 1
        cap.release()
    print(f"Face extraction complete.")
else:
    print(f"No raw videos found at {raw_video_dir}. Skipping preprocessing.")

## Step 7: Train Audio-Visual Fusion Model (PECL — ICCV 2023)

This trains the full audio+visual model with crossmodal attention fusion and parameter-efficient adapters.

In [ ]:
import sys
sys.path.insert(0, 'scripts')

from train_test import train_test
from types import SimpleNamespace

# ============================================================
# Hyperparameters — adjust as needed
# ============================================================
config = SimpleNamespace(
    device='cuda:0',
    device_ID='cuda:0',
    lr=1e-3,
    batch_size=16,
    num_epochs=30,
    seed=2222,
    num_encoders=4,
    adapter=True,
    adapter_type='efficient_conv',
    data_root='data/',
    audio_path='data/audio_files/',
    visual_path='data/face_frames/',
    log='logs',
    protocols=[['train_fold1.csv', 'test_fold1.csv']],
    model_name='DOLOS_',
    model_to_train='fusion',
    fusion_type='cross2',
    multi=False,
)

import torch
torch.manual_seed(config.seed)
config.device = torch.device(config.device_ID)

os.makedirs(config.log, exist_ok=True)

log_name = os.path.join(config.log, f'colab_fusion_seed{config.seed}.txt')
with open(log_name, 'w') as f:
    f.write(f"Optimizer: Adam, LR: {config.lr}\n")
    f.write(f"Batch Size: {config.batch_size}\n")
    f.write(f"Epochs: {config.num_epochs}\n")
    f.write(f"Encoders: {config.num_encoders}\n")
    f.write(f"Adapter: {config.adapter}, Type: {config.adapter_type}\n")
    f.write(f"Fusion: {config.fusion_type}\n")
    f.write("-" * 40 + "\n")

print("Starting training...")
print(f"Config: lr={config.lr}, batch={config.batch_size}, epochs={config.num_epochs}")
print(f"Model: {config.model_to_train}, fusion={config.fusion_type}")
print(f"Adapter: {config.adapter} ({config.adapter_type})")

train_test(log_name, config)

## Step 8: Train 3-Fold Cross-Validation

In [ ]:
import sys
sys.path.insert(0, 'scripts')

from train_test import train_test
from types import SimpleNamespace

all_results = []

for fold in range(1, 4):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/3")
    print(f"{'='*60}")

    config = SimpleNamespace(
        device='cuda:0',
        device_ID='cuda:0',
        lr=1e-3,
        batch_size=16,
        num_epochs=30,
        seed=2222 + fold,
        num_encoders=4,
        adapter=True,
        adapter_type='efficient_conv',
        data_root='data/',
        audio_path='data/audio_files/',
        visual_path='data/face_frames/',
        log='logs',
        protocols=[[f'train_fold{fold}.csv', f'test_fold{fold}.csv']],
        model_name='DOLOS_',
        model_to_train='fusion',
        fusion_type='cross2',
        multi=False,
    )

    import torch
    torch.manual_seed(config.seed)
    config.device = torch.device(config.device_ID)

    os.makedirs(config.log, exist_ok=True)
    log_name = os.path.join(config.log, f'colab_fold{fold}.txt')
    with open(log_name, 'w') as f:
        f.write(f"Fold {fold}\nLR: {config.lr}\nBatch: {config.batch_size}\nEpochs: {config.num_epochs}\n")

    train_test(log_name, config)

print("\n" + "="*60)
print("ALL FOLDS COMPLETE")
print("="*60)

# Print all results
for fold in range(1, 4):
    log_file = os.path.join('logs', f'colab_fold{fold}.txt')
    if os.path.exists(log_file):
        print(f"\n--- Fold {fold} Results ---")
        with open(log_file) as f:
            print(f.read())

## Step 9: Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import re

# Parse training logs
def parse_log(log_path):
    train_accs, test_accs, train_aucs, test_aucs = [], [], [], []
    with open(log_path) as f:
        for line in f:
            match = re.search(
                r'train_acc ([\d.]+).*train_auc:([\d.]+).*test_acc ([\d.]+).*test_auc:([\d.]+)', line)
            if match:
                train_accs.append(float(match.group(1)))
                train_aucs.append(float(match.group(2)))
                test_accs.append(float(match.group(3)))
                test_aucs.append(float(match.group(4)))
    return train_accs, test_accs, train_aucs, test_aucs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for fold in range(1, 4):
    log_file = os.path.join('logs', f'colab_fold{fold}.txt')
    if os.path.exists(log_file):
        train_acc, test_acc, train_auc, test_auc = parse_log(log_file)
        if train_acc:
            axes[0].plot(train_acc, label=f'Fold {fold} Train', linestyle='--')
            axes[0].plot(test_acc, label=f'Fold {fold} Test')
            axes[1].plot(train_auc, label=f'Fold {fold} Train', linestyle='--')
            axes[1].plot(test_auc, label=f'Fold {fold} Test')

axes[0].set_title('Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('AUC per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")

## Step 10: Save Results to Google Drive

In [ ]:
import shutil

# Create output directory on Drive
drive_output = '/content/drive/MyDrive/DeceptionDetection_Results'
os.makedirs(drive_output, exist_ok=True)

# Copy logs
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(drive_output, 'logs'), dirs_exist_ok=True)

# Copy training curves
if os.path.exists('training_curves.png'):
    shutil.copy('training_curves.png', drive_output)

# Copy saved model weights
for f in os.listdir('logs'):
    if f.endswith('.pth'):
        shutil.copy(os.path.join('logs', f), drive_output)

print(f"Results saved to: {drive_output}")
!ls -la "{drive_output}"